# **Taller II. Preparación de datos para la predicción y optimización del consumo energético**

## **1. Definición del problema**

La Universidad Pedagógica y Tecnológica de Colombia (UPTC) opera en cuatro sedes de Boyacá (Tunja, Duitama, Sogamoso y Chiquinquirá) con poblaciones estudiantiles y dinámicas distintas. Cada sede consume energía en cinco sectores: comedores, salones, laboratorios, auditorios y oficinas.

Se dispone de un archivo histórico de lecturas horarias (`consumos_uptc.csv`) que presenta problemas de calidad como:

+ Datos faltantes
+ Registros duplicados
+ Inconsistencias en variables categóricas
+ Valores negativos (físicamente imposibles)
+ Valores atípicos (outliers)

El problema consiste en preparar y transformar el dataset para que sea apto para entrenar modelos de machine learning que predigan el consumo energético y apoyen estrategias de sostenibilidad.

## **2. Objetivo general**

Diseñar y ejecutar un proceso integral de preparación de datos que transforme los registros de consumo energético en un conjunto limpio, consistente y estructurado, apto para modelos de predicción.

## **3. Objetivos específicos**

+ Explorar la estructura, dimensiones y tipos de variables del dataset.
+ Limpiar los datos: nulos, negativos, duplicados e inconsistencias.
+ Analizar estadísticamente las variables energéticas.
+ Transformar variables y construir el conjunto final (variable objetivo y features).

# **Fase I. Carga y exploración inicial de datos**

## Cargar los datos
Al ejecutar la celda se abre un botón **Elegir archivos**: selecciona `consumos_uptc.csv` desde tu computador.

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()          # Se abre el selector de archivos
nombre_archivo = list(uploaded.keys())[0]
print('Archivo cargado:', nombre_archivo)

Convertir el archivo en DataFrame

In [ ]:
df = pd.read_csv(nombre_archivo)

Nombre de variables o columnas

In [ ]:
df.columns

Dimensiones del dataset

In [ ]:
df.shape

El DataFrame contiene **275.387 filas y 26 columnas**: lecturas horarias de 2018 a 2025 de las cuatro sedes.

In [ ]:
display(df.head())

In [ ]:
display(df.tail())

Tipos de variables y valores no nulos

In [ ]:
df.info()

El dataset mezcla variables numéricas (energía por sector, potencia, agua, temperatura, ocupación, CO2), temporales (timestamp, hora, día, mes, trimestre, año), categóricas (sede, sede_id, periodo_academico, dia_nombre) y booleanas (fin de semana, festivo, semana de parciales y de finales). La columna `timestamp` se leyó como texto.

### Valores únicos por sede

In [ ]:
df['sede'].value_counts()

Las cuatro sedes tienen un número muy parecido de registros (alrededor de 68.800 cada una).

In [ ]:
df['sede_id'].value_counts()

`sede_id` debería tener 4 valores, pero tiene **12**: el mismo código aparece escrito como `UPTC_DUI`, `UPTC-DUI` y `uptc_dui`. Es una inconsistencia de formato que se corrige en la Fase II.

In [ ]:
df['periodo_academico'].value_counts()

`periodo_academico` también está inconsistente: aparecen `semestre_1`, `SEMESTRE_1`, `Semestre_1` y `semestre1` para lo mismo, y `vacaciones` suelto junto a `vacaciones_mitad` y `vacaciones_fin`.

### Estadísticas descriptivas

In [ ]:
df.describe().T

Observaciones:

+ `energia_total_kwh` y `co2_kg` tienen mínimos negativos, lo cual no es posible.
+ `agua_litros` y `energia_total_kwh` tienen máximos muy por encima del 75% de los datos (valores atípicos).
+ Varias columnas tienen un `count` menor al total de filas, es decir, tienen nulos.

### Valores nulos

In [ ]:
nulos = df.isnull().sum()
pd.DataFrame({'nulos': nulos, '%': (nulos / len(df) * 100).round(2)})[nulos > 0]

Tienen nulos: energía de comedor y auditorios, agua, temperatura exterior, ocupación y CO2. La ocupación es la más afectada (cerca del 12%).

# **Fase II. Limpieza y preparación inicial**

## Estandarización de categorías inconsistentes

In [ ]:
# sede_id: mayúsculas y guion bajo
df['sede_id'] = df['sede_id'].str.upper().str.replace('-', '_')

# periodo_academico: minúsculas y unificación de etiquetas
df['periodo_academico'] = (df['periodo_academico'].str.lower()
                           .replace({'semestre1': 'semestre_1', 'vacaciones': 'vacaciones_fin'}))

print(df['sede_id'].value_counts(), '\n')
print(df['periodo_academico'].value_counts())

Ahora `sede_id` tiene 4 valores y `periodo_academico` tiene 4 categorías (`semestre_1`, `semestre_2`, `vacaciones_mitad`, `vacaciones_fin`).

## Filtrar la sede DUITAMA

In [ ]:
registros_antes = len(df)
df = df[df['sede'].str.strip().str.upper() == 'DUITAMA'].copy()
registros_despues = len(df)

print('Registros antes del filtrado  :', registros_antes)
print('Registros después del filtrado:', registros_despues)
print('Registros descartados         :', registros_antes - registros_despues)

Al filtrar por Duitama pasamos de **275.387 a 68.862 registros**; se descartan los de Tunja, Sogamoso y Chiquinquirá.

## Valores nulos en la sede Duitama

In [ ]:
nulos = df.isnull().sum()
nulos[nulos > 0]

Estos nulos se imputarán en la Fase III (mediana para numéricas y moda para categóricas).

## Valores negativos

In [ ]:
cols_negativas = ['energia_comedor_kwh', 'energia_salones_kwh', 'energia_laboratorios_kwh',
                  'energia_auditorios_kwh', 'energia_oficinas_kwh', 'agua_litros', 'co2_kg',
                  'energia_total_kwh']   # se añade la energía total

(df[cols_negativas] < 0).sum()

Solo `energia_total_kwh` y `co2_kg` tienen valores negativos; los cinco sectores y el agua no. Una energía o un CO2 negativos no son físicamente válidos, así que se eliminan esos registros.

In [ ]:
filas_negativas = (df[cols_negativas] < 0).any(axis=1)
print('Registros con algún valor negativo:', filas_negativas.sum())

df = df[~filas_negativas].copy()
print('Registros después de eliminar negativos:', len(df))

Se eliminaron **155 registros** con valores negativos (68.862 → 68.707).

## Registros duplicados

In [ ]:
# 1) Duplicados exactos (ignorando el identificador reading_id)
print('Duplicados exactos:', df.drop(columns='reading_id').duplicated().sum())

# 2) Misma sede y misma hora (debería existir una sola lectura por hora y sede)
print('Duplicados por timestamp + sede:', df.duplicated(subset=['timestamp', 'sede']).sum())

In [ ]:
# Ejemplo de registros repetidos
display(df[df.duplicated(subset=['timestamp', 'sede'], keep=False)].sort_values('timestamp').head(6))

No hay filas idénticas, pero hay **220 lecturas repetidas** para la misma sede y hora: tienen los mismos valores de energía y difieren solo en pequeñas variaciones de `co2_kg`. Como solo debe existir una lectura por hora y sede, se conserva la primera.

In [ ]:
print('Antes :', len(df))
df = df.drop_duplicates(subset=['timestamp', 'sede'], keep='first').reset_index(drop=True)
print('Después:', len(df))

Tras eliminar duplicados quedan **68.487 registros**.

In [ ]:
resumen = pd.DataFrame({
    'Etapa': ['Archivo original', 'Sede Duitama', 'Sin negativos', 'Sin duplicados'],
    'Registros': [registros_antes, registros_despues, registros_despues - int(filas_negativas.sum()), len(df)]})
resumen

# **Fase III. Análisis exploratorio de datos (EDA)**

## Estadísticas descriptivas de las variables energéticas

In [ ]:
energia = ['energia_comedor_kwh', 'energia_salones_kwh', 'energia_laboratorios_kwh',
           'energia_auditorios_kwh', 'energia_oficinas_kwh']

df[energia].describe().T

Los **laboratorios** son el sector que más consume (media cercana a 3,6 kWh), seguido de oficinas y salones; comedores y auditorios consumen muy poco. En todos los sectores la media es mayor que la mediana, lo que indica distribuciones sesgadas a la derecha.

In [ ]:
df[energia].skew()

Todas las asimetrías son positivas (mayores a 1); las más marcadas son auditorios y comedores: casi siempre consumen poco y tienen picos ocasionales.

## Histogramas de las variables energéticas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, energia):
    sns.histplot(df[col].dropna(), bins=40, kde=True, color='skyblue', edgecolor='black', ax=ax)
    ax.set_title(col.replace('energia_', '').replace('_kwh', '').capitalize())
    ax.set_xlabel('kWh')
axes.flat[5].axis('off')
plt.tight_layout()
plt.show()

Los histogramas muestran gran concentración de lecturas cerca de cero (noches, fines de semana y vacaciones) y una cola larga hacia la derecha, correspondiente a las horas de mayor actividad académica.

## Diagrama de cajas de la energía por sector

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df[energia], orient='h', color='skyblue', flierprops={'markerfacecolor': 'red', 'marker': 'o', 'markersize': 2})
plt.title('Diagrama de cajas: energía por sector (kWh)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

Los puntos rojos son valores atípicos según el criterio IQR. En consumos horarios muchos de ellos son horas pico reales, por lo que **no se eliminan**; sí conviene revisar los extremos de agua y energía total con el área de infraestructura.

## Imputación de valores nulos
+ Mediana para variables numéricas (robusta ante el sesgo y los atípicos).
+ Moda para variables categóricas.

In [ ]:
nulos_antes = df.isnull().sum()
display(nulos_antes[nulos_antes > 0])

cols_num = df.select_dtypes(include='number').columns.drop('reading_id')
cols_cat = df.select_dtypes(include=['object', 'string']).columns.drop('timestamp')

for c in cols_num:
    df[c] = df[c].fillna(df[c].median())
for c in cols_cat:
    df[c] = df[c].fillna(df[c].mode()[0])

print('Nulos restantes:', df.isnull().sum().sum())

Tras la imputación no quedan valores nulos.

## Matriz de correlación

In [ ]:
cols_corr = ['energia_total_kwh'] + energia + ['potencia_total_kw', 'agua_litros', 'temperatura_exterior_c',
             'ocupacion_pct', 'hora', 'dia_semana', 'mes', 'es_fin_semana', 'es_festivo',
             'es_semana_parciales', 'es_semana_finales', 'co2_kg']

corr = df[cols_corr].astype(float).corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, annot_kws={'size': 7})
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

In [ ]:
corr['energia_total_kwh'].drop('energia_total_kwh').sort_values(key=abs, ascending=False)

Interpretación:

+ `co2_kg` y `potencia_total_kw` tienen correlación muy alta con la energía total porque se derivan de ella (no sirven como predictores).
+ Salones, oficinas y laboratorios correlacionan fuertemente con el total (alrededor de 0,76 a 0,78): son sus componentes. Comedores, de forma moderada.
+ `ocupacion_pct` tiene correlación positiva clara (aprox. 0,66): a más ocupación, más consumo.
+ Las variables de calendario y la temperatura exterior tienen relación más débil.

# **Fase IV. Preparación para modelado**

## Variable objetivo y variables predictoras

+ **Objetivo:** `energia_total_kwh`.
+ **Features:** temperatura exterior, ocupación, hora, día de la semana, mes, trimestre, año, banderas de calendario y periodo académico.
+ **Excluidas:** `reading_id`, `timestamp`, `dia_nombre` (redundantes), `sede` y `sede_id` (constantes, solo Duitama), energía por sector y `potencia_total_kw` (fuga de información: componen o derivan del total) y `co2_kg` y `agua_litros` (se miden al mismo tiempo que el consumo y no estarían disponibles para predecir).

In [ ]:
objetivo = 'energia_total_kwh'

excluir = ['reading_id', 'timestamp', 'sede', 'sede_id', 'dia_nombre',
           'potencia_total_kw', 'co2_kg', 'agua_litros', objetivo] + energia

features = [c for c in df.columns if c not in excluir]
print('Features:', features)

## Transformación de variables

In [ ]:
X = df[features].copy()

# Booleanas a 0/1
X[X.select_dtypes('bool').columns] = X.select_dtypes('bool').astype(int)

# Codificación One-Hot de la variable categórica
X = pd.get_dummies(X, columns=['periodo_academico'], dtype=int)

y = df[objetivo]
df_final = pd.concat([X, y], axis=1)

print('Dimensiones del conjunto final:', df_final.shape)
df_final.head()

In [ ]:
print('Nulos:', df_final.isnull().sum().sum())
print('Negativos en el objetivo:', (df_final[objetivo] < 0).sum())
df_final.dtypes

El conjunto final tiene **68.487 filas y 16 columnas** (15 features + el objetivo), sin nulos, sin negativos y sin duplicados.

## Exportar el conjunto de datos limpio

In [ ]:
df_final.to_csv('dataset_final_duitama.csv', index=False)
files.download('dataset_final_duitama.csv')

## Conclusiones

+ El dataset original tenía inconsistencias de formato, nulos, valores negativos y lecturas repetidas.
+ Para la sede Duitama se pasó de 68.862 registros a 68.487 limpios.
+ El consumo está dominado por laboratorios y se relaciona con la ocupación y el calendario académico.
+ Para entrenar modelos se recomienda dividir los datos **por tiempo** (entrenar con años anteriores y validar con los más recientes) y revisar los valores extremos de agua y energía total.